
# Machine Learning aplicado, Notebook 5
## Riesgo de crédito: predecir un sí o un no con regresión logística

**Curso:** *Machine Learning aplicado* · **Preparado por:** David Díaz, con asistencia de Claude (Anthropic) · **Entorno:** Google Colab

### Un problema distinto

Hasta ahora siempre predijimos un **número**: ventas, retorno, volatilidad, precio. Hoy la pregunta
es de **sí o no**: ¿este cliente va a pagar su crédito o no? Se llama un problema de **clasificación**,
y es probablemente el uso más extendido del machine learning en finanzas: cada banco tiene un modelo que
hace exactamente esto antes de aprobar una tarjeta o un préstamo.

Vamos a usar una sola base, real y pública: **German Credit** (Hofmann, 1994), 1.000 créditos de consumo
de un banco alemán, de los cuales 300 terminaron impagos. Y vamos a usar un solo algoritmo, la
**regresión logística**, de principio a fin. Eso es a propósito: lo que se aprende hoy sobre cómo evaluar
un clasificador, cómo elegir el umbral, qué hacer con clases desbalanceadas y cuánto confiar en una
métrica, vale igual para cualquier otro modelo, y conviene aprenderlo sin tener que aprender otro
algoritmo al mismo tiempo.

Este notebook es el paralelo en Python de tres planillas del curso: `logistica_01_german_credit.xlsx`
(el modelo base), `logistica_02_regularizacion.xlsx` y `logistica_03_desbalanceo.xlsx`. Usa exactamente
los mismos datos, la misma preparación y la misma partición, así que los números tienen que coincidir
celda por celda; si en el Excel corres Solver, deberías llegar a los mismos coeficientes que aquí.

### Qué vas a aprender hoy

1. Por qué una regresión lineal no sirve para un sí o no, y cómo la **función logística** lo arregla.
2. Cómo mide sus errores un modelo que predice probabilidades: la **entropía cruzada**.
3. Cómo se evalúa un clasificador: acierto, **matriz de confusión**, **recall** (eficacia) y
   **precision** (eficiencia), y el trade-off entre las dos.
4. Por qué el **umbral** de 50% es una decisión y no una ley: curva **ROC**, **AUC** y umbral óptimo con costos.
5. Qué es la **regularización** en una logística y qué hace de verdad en esta base.
6. Qué provoca el **desbalanceo** de clases y las cuatro formas de corregirlo.
7. Cuánto se mueve una métrica si cambia la muestra: **validación cruzada** e **intervalos de confianza**.


In [ ]:

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import warnings

from sklearn.datasets import fetch_openml
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (roc_auc_score, roc_curve, log_loss, confusion_matrix, classification_report,
                             accuracy_score, recall_score, precision_score, f1_score)
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold, train_test_split
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings("ignore")
pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 30)



## 1. Los datos y su preparación

German Credit tiene 20 columnas por cliente, tal como las registró el banco en el momento de la solicitud:
cuánto pidió, a cuántos meses, cuánto tenía en la cuenta corriente y en ahorros, historial crediticio, edad,
vivienda, propósito del crédito y otras. La última columna, `class`, dice si el cliente resultó ser buen
pagador (`good`, 700 casos) o mal pagador (`bad`, 300 casos).

La celda siguiente descarga la base desde OpenML. Si tienes el archivo del curso (`german_credit.csv`),
pega su link en `URL_GERMAN` y se usa ese en vez de descargar.


In [ ]:

URL_GERMAN = ""    # <-- pega aquí el link del CSV del curso (german_credit.csv) si lo tienes

def normalizar_link(url):
    '''Convierte un link de Dropbox o Google Drive en un link de descarga directa.'''
    if "dropbox.com" in url:
        return url.replace("dl=0", "dl=1").replace("www.dropbox.com", "dl.dropboxusercontent.com")
    if "drive.google.com" in url and "/d/" in url:
        return "https://drive.google.com/uc?export=download&id=" + url.split("/d/")[1].split("/")[0]
    return url

if URL_GERMAN:
    german = pd.read_csv(normalizar_link(URL_GERMAN))
else:
    german = fetch_openml("credit-g", version=1, as_frame=True).frame
    for columna in german.select_dtypes("category").columns:
        german[columna] = german[columna].astype(str)
german = german.reset_index(drop=True)
german.insert(0, "id", np.arange(1, len(german) + 1))
print("Clientes:", len(german), "  Columnas:", german.shape[1] - 1)
print("Proporción de malos pagadores:", round((german["class"] == "bad").mean(), 3))
german[["id", "duration", "credit_amount", "age", "checking_status", "credit_history", "savings_status", "class"]].head(8)



### Paso 1: transformar y limpiar

Un modelo lineal solo puede multiplicar números por pesos, así que las columnas con palabras se convierten
en columnas 0/1, una por categoría, dejando una categoría fuera como **base** (si todas las dummies de un
grupo valen 0, el cliente está en la base). La variable objetivo pasa a ser $y = 1$ si el cliente fue mal
pagador: ese es el evento que queremos detectar, así que lo llamamos la clase **positiva**.

De las 20 columnas nos quedamos con 9, elegidas por ser las que un analista de crédito miraría primero, que
se convierten en 4 numéricas y 14 columnas 0/1. Es la misma selección del Excel; al final del notebook hay un
ejercicio con las 20.


In [ ]:

T = pd.DataFrame({"id": german["id"], "y": (german["class"] == "bad").astype(int)})
T["duracion"] = german["duration"].astype(float)                     # meses
T["monto"] = german["credit_amount"].astype(float)                   # marcos alemanes (DM)
T["edad"] = german["age"].astype(float)
T["cuota_pct_ingreso"] = german["installment_commitment"].astype(float)   # tramo 1 a 4 de la cuota como % del ingreso

cs = german["checking_status"]                                       # base: no tiene cuenta corriente
T["cta_negativa"] = (cs == "<0").astype(int)
T["cta_0_200"] = (cs == "0<=X<200").astype(int)
T["cta_sobre_200"] = (cs == ">=200").astype(int)
ch = german["credit_history"]                                        # base: créditos vigentes al día
T["hist_critico"] = (ch == "critical/other existing credit").astype(int)
T["hist_atrasos"] = (ch == "delayed previously").astype(int)
T["hist_sin_creditos"] = ch.isin(["all paid", "no credits/all paid"]).astype(int)
sv = german["savings_status"]                                        # base: ahorro menor a 100 DM
T["ahorro_100_1000"] = sv.isin(["100<=X<500", "500<=X<1000"]).astype(int)
T["ahorro_sobre_1000"] = (sv == ">=1000").astype(int)
T["ahorro_desconocido"] = (sv == "no known savings").astype(int)
hs = german["housing"]                                               # base: vivienda propia
T["vivienda_arriendo"] = (hs == "rent").astype(int)
T["vivienda_gratis"] = (hs == "for free").astype(int)
pu = german["purpose"]                                               # base: otros propósitos
T["proposito_auto"] = pu.isin(["new car", "used car"]).astype(int)
T["proposito_radio_tv"] = (pu == "radio/tv").astype(int)
T["proposito_muebles"] = (pu == "furniture/equipment").astype(int)

XCOLS = [c for c in T.columns if c not in ("id", "y")]
NUM = ["duracion", "monto", "edad", "cuota_pct_ingreso"]
print(len(XCOLS), "columnas explicativas:", XCOLS)
T.head()



### Paso 2: separar training y testing, y escalar solo con training

Las 1.000 filas se separan **antes** de hacer cualquier otra cosa: 700 para entrenar (training) y 300 para
medir (testing), al azar pero manteniendo la proporción de malos pagadores en los dos grupos (se llama
partición **estratificada**). La semilla fija hace que salga la misma partición que en el Excel.

Después escalamos las cuatro columnas numéricas: a cada una le restamos su promedio y la dividimos por su
desviación estándar, igual que en los notebooks anteriores. Las columnas 0/1 no se escalan, para que su
coeficiente se lea como el efecto de pertenecer a esa categoría. El detalle que importa: **el promedio y la
desviación se calculan solo con las 700 filas de training**, y con esos números se escalan las 300 de
testing. Si usáramos las 1.000, testing estaría influyendo en el modelo, aunque sea de forma mínima. Es la
primera regla de higiene del curso: testing no participa en nada que sea aprender.


In [ ]:

rng = np.random.default_rng(2026)
conjunto = np.array(["training"] * len(T), dtype=object)
for clase in (0, 1):
    idx = np.where(T["y"].values == clase)[0]
    rng.shuffle(idx)
    conjunto[idx[: int(round(0.3 * len(idx)))]] = "testing"
T["conjunto"] = conjunto
tr = T["conjunto"] == "training"
te = ~tr

media = T.loc[tr, NUM].mean()
desv = T.loc[tr, NUM].std(ddof=0)      # desviación poblacional, la misma que usa StandardScaler
S = T.copy()
S[NUM] = (T[NUM] - media) / desv

X_tr, y_tr = S.loc[tr, XCOLS].values, S.loc[tr, "y"].values
X_te, y_te = S.loc[te, XCOLS].values, S.loc[te, "y"].values
print("training:", tr.sum(), "clientes,", y_tr.sum(), "malos pagadores  |  testing:", te.sum(), "clientes,", y_te.sum(), "malos pagadores")
pd.DataFrame({"promedio training": media.round(3), "desviación training": desv.round(3)})



## 2. De la recta a la logística

### El intento ingenuo

La primera idea es usar la regresión lineal de siempre con $y = 1$ si el cliente no pagó y $y = 0$ si pagó:

$$\hat{y} = \theta_0 + \theta_1 x_1 + \theta_2 x_2 + \dots + \theta_{18} x_{18}$$

donde $x_1, \dots, x_{18}$ son las columnas preparadas y $\theta_0, \dots, \theta_{18}$ los pesos que el
modelo ajusta. Funciona a medias. Una recta no tiene tope: para algunos clientes predice un número negativo y,
para un cliente extremo, un número mayor que 1; ninguno de los dos significa nada como probabilidad. Veámoslo.


In [ ]:

lineal = LinearRegression().fit(X_tr, y_tr)
y_lin = lineal.predict(S[XCOLS].values)
print("Predicción lineal mínima:", round(y_lin.min(), 3), "  máxima:", round(y_lin.max(), 3))
print("Clientes con predicción fuera de [0, 1]:", int(((y_lin < 0) | (y_lin > 1)).sum()), "de", len(y_lin))

# un cliente hipotético extremo: crédito de 72 meses por 18.000 DM, 21 años, cuota alta, cuenta negativa, atrasos, arriendo
hip = pd.Series(0.0, index=XCOLS)
hip["duracion"], hip["monto"], hip["edad"], hip["cuota_pct_ingreso"] = 72, 18000, 21, 4
hip["cta_negativa"] = hip["hist_atrasos"] = hip["vivienda_arriendo"] = 1
hip[NUM] = (hip[NUM] - media) / desv
print("Predicción lineal del cliente hipotético:", round(float(lineal.predict(hip.values.reshape(1, -1))[0]), 3))



### La función logística

La solución tiene más de 70 años y se llama **regresión logística** (Berkson, 1944). La receta: se arma la
misma combinación lineal, pero en vez de leerla directo como predicción, se la pasa por una función que la
aplasta al rango entre 0 y 1.

$$z = \theta_0 + \theta_1 x_1 + \dots + \theta_{18} x_{18} \qquad\qquad p = \frac{1}{1 + e^{-z}}$$

Lo que significa cada cosa: $z$ es el **puntaje** del cliente, un número cualquiera, sin límites, que resume
su riesgo; $p$ es la **probabilidad estimada** de que no pague. La segunda fórmula es la **función logística**
o sigmoide: con $z = 0$ da $p = 0{,}5$; con $z = 3$ da $0{,}95$; con $z = -3$ da $0{,}05$; nunca toca 0 ni 1.
Se le llama función de **activación** o de escalamiento: la recta puede dar lo que quiera, la sigmoide lo
trae al rango de una probabilidad. (Existe una prima, la **probit**, que usa la normal acumulada y da una curva
casi idéntica; la logística se prefiere porque su derivada es más simple y porque $z$ resulta ser exactamente
el logaritmo de las chances, $\ln(p/(1-p))$, lo que hace interpretables los pesos.)

Con $p$ en mano aparece la segunda pieza: el **umbral**. Para decidir sí o no hay que colapsar la probabilidad:
si $p$ supera el umbral, predecimos "no paga"; si no, "paga". Por ahora el umbral es 0,5, el valor obvio. Más
adelante vamos a cuestionarlo.


In [ ]:

z = np.linspace(-6, 6, 121)
from scipy.stats import norm
fig = go.Figure()
fig.add_scatter(x=z, y=1 / (1 + np.exp(-z)), name="logística 1/(1+e^-z)")
fig.add_scatter(x=z, y=norm.cdf(z), name="probit (normal acumulada)", line=dict(dash="dash"))
fig.update_layout(title="La función logística (y la probit): cualquier puntaje se convierte en una probabilidad",
                  xaxis_title="puntaje z", yaxis_title="p", height=380)
fig.show()



## 3. Cómo mide sus errores un modelo que predice probabilidades

Con una recta usábamos el error cuadrático. Con probabilidades hay algo mejor. Si un cliente resultó malo
($y = 1$) y el modelo le había dado $p = 0{,}9$, se equivocó poco; si le había dado $p = 0{,}1$, se equivocó
mucho. La pérdida $-\ln(p)$ captura eso: vale 0,105 en el primer caso y 2,303 en el segundo, y crece sin
límite a medida que $p$ se acerca a 0. Para un cliente bueno ($y = 0$) la pérdida es $-\ln(1 - p)$, la
imagen espejo. Escritas juntas, para el cliente $i$:

$$\text{pérdida}_i = -\left[ y_i \ln(p_i) + (1 - y_i) \ln(1 - p_i) \right]$$

Se llama **entropía cruzada binaria** o *log loss*. La **función de costo** del modelo es la suma (o el
promedio) de esa pérdida sobre los clientes de training, y entrenar es encontrar los pesos $\theta$ que la
minimizan. Por qué el logaritmo y no $(y - p)^2$: porque el logaritmo castiga la seguridad equivocada. Decir
99% y fallar cuesta 4,6; decir 60% y fallar cuesta 0,9. Un modelo que minimiza esta pérdida aprende a no ser
más seguro de lo que los datos justifican.


In [ ]:

p_grid = np.linspace(0.005, 0.995, 199)
fig = go.Figure()
fig.add_scatter(x=p_grid, y=-np.log(p_grid), name="si y = 1: −ln(p)")
fig.add_scatter(x=p_grid, y=-np.log(1 - p_grid), name="si y = 0: −ln(1 − p)")
fig.add_scatter(x=p_grid, y=(1 - p_grid) ** 2, name="error cuadrático si y = 1: (1 − p)²", line=dict(dash="dot"))
fig.update_layout(title="Pérdida según la probabilidad predicha", xaxis_title="p predicha", yaxis_title="pérdida", height=380)
fig.show()

# dos modelos sobre los mismos cuatro clientes: el que se equivoca con confianza paga caro
ejemplo = pd.DataFrame({"clase real y": [0, 0, 1, 1], "modelo A: p": [0.01, 0.80, 0.90, 0.10], "modelo B: p": [0.01, 0.05, 0.90, 0.80]})
for m in ("modelo A: p", "modelo B: p"):
    ejemplo[m.replace(": p", ": pérdida")] = -np.where(ejemplo["clase real y"] == 1, np.log(ejemplo[m]), np.log(1 - ejemplo[m]))
print(ejemplo.round(3))
print("Pérdida total A:", round(ejemplo["modelo A: pérdida"].sum(), 3), "  B:", round(ejemplo["modelo B: pérdida"].sum(), 3))



## 4. Entrenar el modelo

En el Excel los pesos se encuentran con Solver minimizando la suma de pérdidas en training. Aquí lo hace
`LogisticRegression` de scikit-learn con un algoritmo numérico más fino, pero es el mismo problema: como la
entropía cruzada de una logística tiene un solo mínimo, los dos deberían llegar a los mismos pesos. Le pedimos
`penalty=None` porque scikit-learn regulariza por defecto, y la regularización la vemos recién en la sección 8.


In [ ]:

logit = LogisticRegression(penalty=None, max_iter=5000).fit(X_tr, y_tr)
p_tr = logit.predict_proba(X_tr)[:, 1]
p_te = logit.predict_proba(X_te)[:, 1]

pesos = pd.Series(np.r_[logit.intercept_, logit.coef_[0]], index=["intercepto"] + XCOLS, name="theta")
print(pesos.round(4))
print()
print("Suma de pérdidas en training (el objetivo de Solver):", round(log_loss(y_tr, p_tr) * len(y_tr), 2))
print("Pérdida promedio training:", round(log_loss(y_tr, p_tr), 4), "  testing:", round(log_loss(y_te, p_te), 4))
print("p mínima y máxima en testing:", round(p_te.min(), 3), round(p_te.max(), 3), " (siempre dentro de (0, 1))")



Cómo leer los pesos: están en unidades de $z$, es decir, de logaritmo de chances. `cta_negativa` con 1,55
dice que tener la cuenta corriente en rojo multiplica las chances de no pagar por $e^{1{,}55} \approx 4{,}7$
respecto de no tener cuenta (la categoría base), con todo lo demás igual. `duracion` con 0,33 dice que un
crédito una desviación estándar más largo (unos 12 meses) multiplica las chances por 1,4. Los negativos
protegen: ahorro conocido y alto, historial crítico (que en esta base significa "ya tiene otros créditos y
los está pagando"), edad.



## 5. Cómo evaluamos el modelo

### El acierto y lo que esconde

El punto de partida es el **acierto** (*accuracy*): qué fracción de los clientes quedó bien clasificada con
el umbral de 0,5. Pero esconde lo que importa: en esta base, un modelo que dijera "todos pagan" acertaría el
70% sin haber aprendido nada, y no atraparía a ningún mal pagador.

### La matriz de confusión

La matriz separa los cuatro casos. Con positivo = mal pagador (el evento que queremos detectar):

- **TP** (verdadero positivo): era malo y lo marcamos.
- **FP** (falso positivo): era bueno y lo marcamos.
- **TN** (verdadero negativo): era bueno y lo dejamos pasar.
- **FN** (falso negativo): era malo y lo dejamos pasar.

### Recall y precision: eficacia y eficiencia

De la matriz salen las dos métricas que hay que saber leer:

$$\text{recall} = \frac{TP}{TP + FN} \qquad\qquad \text{precision} = \frac{TP}{TP + FP}$$

**Recall**: de todos los malos pagadores que había, qué fracción atrapé. Es una medida de **eficacia**.
**Precision**: de todos los que marqué como malos, qué fracción lo era de verdad. Es una medida de
**eficiencia**, porque cada marca en falso cuesta (un buen cliente rechazado, una llamada de cobranza inútil).
**F1** es el promedio armónico de las dos, útil como resumen cuando no se sabe cuál importa más.

Cada clase tiene sus propios recall y precision; `classification_report` de scikit-learn los muestra juntos
(el recall de la clase 0 también se llama especificidad). Las medimos en training y en testing; la que cuenta
es la de testing.


In [ ]:

UMBRAL = 0.5
def matriz(y_real, p, umbral=UMBRAL):
    pred = (p > umbral).astype(int)
    cm = confusion_matrix(y_real, pred, labels=[0, 1])
    return pd.DataFrame(cm, index=["real 0 (paga)", "real 1 (no paga)"], columns=["predicho 0", "predicho 1"])

for nombre, yy, pp in [("TRAINING", y_tr, p_tr), ("TESTING", y_te, p_te)]:
    pred = (pp > UMBRAL).astype(int)
    print(f"=== {nombre}: acierto {accuracy_score(yy, pred):.3f}   (modelo 'todos pagan': {1 - yy.mean():.3f})")
    print(matriz(yy, pp))
    print(classification_report(yy, pred, target_names=["clase 0 (paga)", "clase 1 (no paga)"], digits=3))



### El trade-off entre eficacia y eficiencia

Con umbral 0,5 el modelo atrapa al 57% de los malos pagadores de testing (recall) y de los que marca, el 65%
lo es de verdad (precision). Ese par tiene un trade-off que depende del negocio, y no hay una respuesta
general:

| Caso | Qué cuesta un falso negativo (no marcar) | Qué cuesta un falso positivo (marcar de más) | Qué se privilegia |
|---|---|---|---|
| Riesgo de crédito | Aprobar a quien no paga: se pierde el capital | Rechazar a quien pagaba: se pierde el margen | Recall (eficacia), con costos asimétricos |
| Churn | No contactar a quien se iba: se pierde el cliente | Contactar a quien se quedaba: una llamada, un descuento | Recall, pero el presupuesto de retención es finito, así que precision importa más de lo que parece |
| Fraude | Dejar pasar una operación fraudulenta: la pérdida completa | Bloquear una legítima: un cliente honesto molesto | Precision (eficiencia) mantiene tolerable la fricción |

La palanca para moverse entre los dos es el umbral, y de eso trata la sección siguiente.



## 6. Por qué 50% y no otro número: mover el umbral, la curva ROC y el AUC

El umbral no es parte del modelo; es una decisión. La tabla siguiente recalcula la matriz de confusión sobre
testing para cada umbral entre 0 y 1 y muestra cómo se mueven recall, precision y F1. Bajar el umbral atrapa
más malos pagadores (sube recall) a costa de marcar más buenos (baja precision); subirlo hace lo contrario. No
hay umbral que mejore las dos a la vez: ese es el trade-off, ahora con números.


In [ ]:

def metricas_por_umbral(y_real, p, umbrales):
    filas = []
    for u in umbrales:
        pred = (p > u).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_real, pred, labels=[0, 1]).ravel()
        rec = tp / (tp + fn) if tp + fn else 0.0
        pre = tp / (tp + fp) if tp + fp else 0.0
        filas.append(dict(umbral=u, TP=tp, FP=fp, TN=tn, FN=fn, acierto=(tp + tn) / len(y_real), recall=rec, precision=pre,
                          F1=2 * rec * pre / (rec + pre) if rec + pre else 0.0, tasa_FP=fp / (fp + tn) if fp + tn else 0.0))
    return pd.DataFrame(filas)

umbrales = np.abs(np.round(np.arange(1.0, -0.001, -0.05), 2))
tabla_u = metricas_por_umbral(y_te, p_te, umbrales)
print(tabla_u.round(3).to_string(index=False))

fig = go.Figure()
for col in ("recall", "precision", "F1"):
    fig.add_scatter(x=tabla_u["umbral"], y=tabla_u[col], name=col, mode="lines+markers")
fig.update_layout(title="Recall, precision y F1 en testing según el umbral", xaxis_title="umbral", yaxis_title="", height=380)
fig.show()



### La curva ROC es esa misma tabla dibujada

En el eje horizontal va la **tasa de falsos positivos** ($FP / (FP + TN)$: qué fracción de los buenos
marqué) y en el vertical el **recall** (también llamado tasa de verdaderos positivos). Cada punto es un
umbral; recorrer la curva es mover el umbral desde 1 (nadie marcado, esquina inferior izquierda) hasta 0
(todos marcados, esquina superior derecha). Un modelo que adivina al azar sigue la diagonal; uno perfecto sube
por el borde izquierdo.

El **área bajo la curva** (AUC) resume qué tan lejos de la diagonal está. Se puede calcular a mano como suma
de trapecios sobre los 21 puntos de la tabla; scikit-learn usa los 300 scores de testing como umbrales, así
que obtiene un área un poco más fina. Las dos se muestran.

Una lectura complementaria del AUC que conviene conocer: es la probabilidad de que, tomando al azar un mal
pagador y un buen pagador, el modelo le haya asignado más riesgo al malo. Sirve para comparar modelos sin
fijar umbral, pero esconde el trade-off: cuando dos curvas ROC se cruzan, un solo número no decide nada, y
hay que mirar la curva en la zona de umbrales que el negocio va a usar.


In [ ]:

# AUC por trapecios sobre la tabla (los umbrales van de 1 a 0, así que la tasa de FP crece)
x_, y_ = tabla_u["tasa_FP"].values, tabla_u["recall"].values
auc_trapecios = float(np.sum((x_[1:] - x_[:-1]) * (y_[1:] + y_[:-1]) / 2))
auc_sklearn = roc_auc_score(y_te, p_te)
print("AUC por trapecios (21 umbrales):", round(auc_trapecios, 4), "   AUC scikit-learn (300 umbrales):", round(auc_sklearn, 4))
print("AUC en training:", round(roc_auc_score(y_tr, p_tr), 4))

fpr, tpr, _ = roc_curve(y_te, p_te)
fig = go.Figure()
fig.add_scatter(x=fpr, y=tpr, name=f"modelo (AUC {auc_sklearn:.3f})", mode="lines")
fig.add_scatter(x=x_, y=y_, name="los 21 umbrales de la tabla", mode="markers", marker=dict(size=8))
fig.add_scatter(x=[0, 1], y=[0, 1], name="azar (AUC 0,5)", line=dict(dash="dash"))
fig.update_layout(title="Curva ROC en testing", xaxis_title="tasa de falsos positivos", yaxis_title="recall (tasa de verdaderos positivos)", height=450, width=550)
fig.show()



Un detalle honesto: aquí el AUC de testing (0,82) sale mayor que el de training (0,78). No es lo habitual y
no significa que el modelo generalice mejor de lo que aprendió: con 300 clientes de testing el AUC tiene un
error muestral de varios puntos, y en la sección 10 lo vamos a medir.



## 7. Si sé cuánto cuesta cada error: el umbral óptimo

Si conozco el costo de cada resultado, el umbral deja de ser una discusión y pasa a ser un cálculo. La propia
base German Credit trae una matriz de costos: aprobar a un mal pagador ($FN$) cuesta 5, rechazar a un buen
pagador ($FP$) cuesta 1, y acertar no cuesta nada.

Hay dos formas de encontrar el umbral, y responden a preguntas distintas.

**La fórmula.** Para un cliente con probabilidad $p$ de no pagar, marcarlo cuesta en promedio
$(1 - p) \cdot c_{FP}$ (si en realidad pagaba) y no marcarlo cuesta $p \cdot c_{FN}$ (si en realidad no
pagaba). Conviene marcarlo cuando $p \cdot c_{FN} > (1 - p) \cdot c_{FP}$, es decir, cuando

$$p > \frac{c_{FP}}{c_{FP} + c_{FN}}$$

Con 1 y 5 da 0,167. Fíjate en que no aparece ningún dato: la regla es la misma para cualquier modelo y
cualquier muestra, y es exacta bajo una sola condición, que $p$ sea una probabilidad bien **calibrada**, o sea,
que de los clientes a los que el modelo les da $p = 0{,}3$ efectivamente no pague el 30%. (Con beneficios por
acertar la fórmula es $(c_{FP} - b_{TN}) / ((c_{FP} - b_{TN}) + (c_{FN} - b_{TP}))$.)

**La tabla.** Recorrer los umbrales, calcular el costo total en testing con la matriz de confusión de cada
uno, y quedarse con el mínimo. No supone calibración y sirve con cualquier estructura de costos, incluso si el
costo depende del cliente. A cambio, optimiza contra una muestra de 300, así que hereda su ruido.


In [ ]:

c_TP, c_FP, c_TN, c_FN = 0, 1, 0, 5
umbral_formula = (c_FP - c_TN) / ((c_FP - c_TN) + (c_FN - c_TP))
tabla_u["costo"] = tabla_u["TP"] * c_TP + tabla_u["FP"] * c_FP + tabla_u["TN"] * c_TN + tabla_u["FN"] * c_FN
fila_min = tabla_u.loc[tabla_u["costo"].idxmin()]
print("Umbral óptimo por fórmula:", round(umbral_formula, 3))
print("Umbral con menor costo en la tabla:", fila_min["umbral"], " costo:", int(fila_min["costo"]), "  (costo con umbral 0,5:", int(tabla_u.loc[tabla_u.umbral == 0.5, "costo"].iloc[0]), ", 'todos pagan':", int(tabla_u.loc[tabla_u.umbral == 1.0, "costo"].iloc[0]), ")")
fig = px.line(tabla_u, x="umbral", y="costo", markers=True, title="Costo total en testing según el umbral (FN = 5, FP = 1)")
fig.add_vline(x=umbral_formula, line_dash="dash", annotation_text="fórmula 0,167")
fig.update_layout(height=360)
fig.show()



La tabla dice 0,25 y la fórmula 0,167. Las dos causas posibles son independientes y dejan huellas distintas.

**Mala calibración** es un problema del modelo: si a los clientes con $p = 0{,}2$ en realidad les deja de
pagar el 30%, la fórmula apunta al lugar equivocado y la tabla lo corrige. Se diagnostica agrupando testing
por deciles de $p$ y comparando el $p$ promedio de cada grupo con la fracción real de malos pagadores.

**Variación muestral** es un problema de los datos: con 300 clientes, mover el umbral de 0,25 a 0,30 cambia la
clasificación de unos 15, y el costo de esos 15 depende de quién cayó en la muestra. El valle alrededor del
mínimo es plano (136, 138, 141) y con otra muestra el óptimo salta. Se diagnostica remuestreando, que es lo
que hacemos en la sección 10.

Veamos la calibración.


In [ ]:

cal = pd.DataFrame({"p": p_te, "y": y_te})
cal["grupo"] = pd.qcut(cal["p"], 5, labels=["20% menos riesgoso", "2", "3", "4", "20% más riesgoso"])
calib = cal.groupby("grupo", observed=True).agg(clientes=("y", "size"), p_promedio=("p", "mean"), fraccion_real_malos=("y", "mean"))
print(calib.round(3))
print("\nSi el modelo está calibrado, las dos últimas columnas se parecen. Con 60 clientes por grupo, diferencias de 0,05 son ruido.")



## 8. Regularización: cobrarle al modelo por el tamaño de los pesos

La idea viene de las clases de regresión lineal: cuando un modelo tiene libertad para mover muchos pesos,
tiende a ajustarse al ruido de training, y una forma de frenarlo es agregar a la función de costo un término
que crece con el tamaño de los pesos. Con la logística es igual de fácil:

$$J(\theta) = \sum_i \text{pérdida}_i + \text{penalización}(\theta)$$

- **Ridge (L2)**: penalización $= \frac{\lambda}{2} \sum_j \theta_j^2$. Encoge todos los pesos hacia cero, más a los grandes, pero ninguno llega a cero.
- **Lasso (L1)**: penalización $= \lambda \sum_j |\theta_j|$. También encoge, pero deja pesos exactamente en cero: elimina variables.
- **Elastic net**: mezcla, $\lambda \left[ \alpha \sum |\theta_j| + \frac{1 - \alpha}{2} \sum \theta_j^2 \right]$, con $\alpha$ la fracción L1.

En todas, $\lambda$ dice cuánto cuesta el tamaño, y el intercepto queda fuera (no dice nada sobre ninguna
variable). En scikit-learn el parámetro es $C = 1/\lambda$. Vamos a recorrer $\lambda$ y ver qué pasa con los
pesos y con el desempeño en testing.


In [ ]:

def ajustar(pen, lam, alpha=0.5):
    if lam == 0:
        m = LogisticRegression(penalty=None, max_iter=10000)
    elif pen == "l2":
        m = LogisticRegression(penalty="l2", C=1 / lam, max_iter=10000)
    elif pen == "l1":
        m = LogisticRegression(penalty="l1", C=1 / lam, solver="saga", max_iter=50000, tol=1e-8)
    else:
        m = LogisticRegression(penalty="elasticnet", C=1 / lam, l1_ratio=alpha, solver="saga", max_iter=50000, tol=1e-8)
    m.fit(X_tr, y_tr)
    pt, pp = m.predict_proba(X_tr)[:, 1], m.predict_proba(X_te)[:, 1]
    return m, dict(lam=lam, perdida_train=log_loss(y_tr, pt), perdida_test=log_loss(y_te, pp), auc_train=roc_auc_score(y_tr, pt),
                   auc_test=roc_auc_score(y_te, pp), pesos_en_cero=int((np.abs(m.coef_[0]) < 1e-6).sum()), suma_abs_pesos=np.abs(m.coef_[0]).sum())

LAMS = [0, 1, 3, 10, 30, 100, 300]
resultados, trayectorias = {}, {}
for pen in ("l2", "l1", "en"):
    filas, tray = [], []
    for lam in LAMS:
        m, r = ajustar(pen, lam); filas.append(r); tray.append(m.coef_[0])
    resultados[pen] = pd.DataFrame(filas).set_index("lam")
    trayectorias[pen] = pd.DataFrame(np.array(tray), index=LAMS, columns=XCOLS)

for pen, nombre in [("l2", "Ridge (L2)"), ("l1", "Lasso (L1)"), ("en", "Elastic net (alpha 0,5)")]:
    print(f"=== {nombre}")
    print(resultados[pen].round(4).to_string())
    print()


In [ ]:

fig = go.Figure()
for col in XCOLS:
    fig.add_scatter(x=[str(l) for l in LAMS], y=trayectorias["l1"][col], name=col, mode="lines+markers")
fig.update_layout(title="Lasso: cada peso según lambda (las variables se van apagando de a una)", xaxis_title="lambda", yaxis_title="theta", height=480)
fig.show()



Lectura honesta, que conviene decir tal cual: en esta base **regularizar no mejora la predicción**. La
pérdida de testing sube con $\lambda$ en vez de bajar, y el AUC de testing se mantiene en 0,82 hasta
$\lambda = 10$ y después cae. Con 700 clientes y 18 variables, la logística sin regularizar no estaba
sobreajustando; no había nada que frenar. Lo que la regularización sí entrega aquí es un modelo más chico
(lasso con $\lambda = 10$ apaga 7 de las 18 variables y pierde una centésima de AUC) y pesos más estables ante
cambios de muestra. Regularizar sí mejora testing cuando hay muchas variables respecto de las filas, o
variables muy correlacionadas, o polinomios de grado alto; eso se ve en el material avanzado.



## 9. Clases desbalanceadas: qué provoca y cómo se corrige

En German Credit el 30% de los clientes es mal pagador. La entropía cruzada suma el error de cada cliente por
igual, así que el modelo se preocupa más de acertarle a los 490 buenos de training que a los 210 malos: por
eso el recall de la clase 0 es 0,87 y el de la clase 1, que es la que importa, 0,57. En bases más
desbalanceadas (fraude con 1% de casos) el efecto es peor: el modelo aprende a decir "no" siempre y acierta el
99%.

Las soluciones cambian una de dos cosas: cuánto pesa cada cliente en la función de costo, o qué clientes
entran en training.

- **Class weights**: la pérdida de cada fila se multiplica por un peso que depende de su clase, típicamente
  $n / (2 \cdot n_{clase})$ para que las dos clases sumen lo mismo (`class_weight="balanced"`).
- **Submuestreo**: se tiran al azar buenos pagadores hasta igualar a los malos, y se entrena con menos filas.
- **Sobremuestreo**: se repiten las filas de la clase minoritaria hasta que las dos clases pesen lo mismo. En el caso
  general la cuenta no da un número entero: con 490 buenos y 210 malos habría que repetir cada malo 490/210 = 2,33
  veces. Se resuelve de dos formas equivalentes: muestrear con reposición 490 filas entre los 210 malos (unos salen
  dos veces y otros tres; es lo que hace `RandomOverSampler` de `imbalanced-learn`), o darle a cada malo un peso de
  2,33 en la función de costo (`sample_weight`), que es exactamente class weights. Repetir una fila k veces siempre
  equivale a un peso k, entero o no. Abajo van las dos versiones: la simple, con cada malo dos veces, y la genérica.
- **SMOTE** (Chawla y otros, 2002): no repite, inventa. Toma un mal pagador, busca sus vecinos más cercanos
  entre los malos pagadores, y crea un cliente sintético en un punto al azar del segmento que los une:
  $x_{nuevo} = x_a + u \cdot (x_b - x_a)$ con $u$ entre 0 y 1. Está en la librería `imbalanced-learn`; aquí
  lo escribimos a mano en diez líneas para que se vea qué hace.

Todo se hace **solo sobre training**: testing se queda con sus 300 clientes reales.


In [ ]:

def evaluar(m, nombre, umbral=0.5):
    p = m.predict_proba(X_te)[:, 1]
    pred = (p > umbral).astype(int)
    return dict(modelo=nombre, umbral=umbral, acierto=accuracy_score(y_te, pred), recall_1=recall_score(y_te, pred), precision_1=precision_score(y_te, pred),
                F1_1=f1_score(y_te, pred), recall_0=recall_score(y_te, pred, pos_label=0), AUC=roc_auc_score(y_te, p), p_promedio=p.mean())

n1, n0 = int(y_tr.sum()), int((y_tr == 0).sum())
comparacion = [evaluar(logit, "sin balancear")]

# 1) class weights
cw = LogisticRegression(penalty=None, max_iter=5000, class_weight="balanced").fit(X_tr, y_tr)
print("pesos 'balanced': clase 1 =", round(len(y_tr) / (2 * n1), 3), " clase 0 =", round(len(y_tr) / (2 * n0), 3))
comparacion.append(evaluar(cw, "class weights balanced"))

# 2) submuestreo: todos los malos + n1 buenos al azar
rng_b = np.random.default_rng(11)
idx_buenos = rng_b.choice(np.where(y_tr == 0)[0], size=n1, replace=False)
idx_sub = np.r_[np.where(y_tr == 1)[0], idx_buenos]
sub = LogisticRegression(penalty=None, max_iter=5000).fit(X_tr[idx_sub], y_tr[idx_sub])
comparacion.append(evaluar(sub, f"submuestreo {n1} + {n1}"))

# 3) sobremuestreo simple: cada malo pagador dos veces (equivale a sample_weight 2; 420 malos contra 490 buenos)
over = LogisticRegression(penalty=None, max_iter=5000).fit(X_tr, y_tr, sample_weight=np.where(y_tr == 1, 2.0, 1.0))
comparacion.append(evaluar(over, "sobremuestreo, malos x2"))
# 3b) sobremuestreo genérico: muestrear con reposición n0 malos (490 contra 490), como RandomOverSampler
rng_o = np.random.default_rng(7)
idx_over = np.r_[np.where(y_tr == 0)[0], rng_o.choice(np.where(y_tr == 1)[0], size=n0, replace=True)]
over_g = LogisticRegression(penalty=None, max_iter=5000).fit(X_tr[idx_over], y_tr[idx_over])
comparacion.append(evaluar(over_g, f"sobremuestreo aleatorio {n0} + {n0}"))

# 4) SMOTE a mano: sintéticos hasta igualar 490 y 490
X_mal = X_tr[y_tr == 1]
vecinos = NearestNeighbors(n_neighbors=6).fit(X_mal)
rng_s = np.random.default_rng(5)
sinteticos = []
for _ in range(n0 - n1):
    i = rng_s.integers(len(X_mal))
    j = rng_s.choice(vecinos.kneighbors(X_mal[i:i + 1], return_distance=False)[0][1:])
    u = rng_s.uniform()
    sinteticos.append(X_mal[i] + u * (X_mal[j] - X_mal[i]))
X_sm = np.vstack([X_tr, np.array(sinteticos)]); y_sm = np.r_[y_tr, np.ones(n0 - n1, int)]
smote = LogisticRegression(penalty=None, max_iter=5000).fit(X_sm, y_sm)
comparacion.append(evaluar(smote, f"SMOTE, {n0} + {n0}"))

# y el modelo sin balancear con otro umbral
comparacion.append(evaluar(logit, "sin balancear", umbral=0.30))
comparacion.append(evaluar(logit, "sin balancear", umbral=0.25))
print(pd.DataFrame(comparacion).round(3).to_string(index=False))



Los cinco métodos suben el recall de la clase 1 de 0,57 a entre 0,82 y 0,87 y bajan la precision de 0,65 a
entre 0,50 y 0,56: ninguno crea información, todos mueven el punto de operación hacia atrapar más malos
pagadores. Las dos versiones del sobremuestreo cuentan la misma historia: la simple (malos x2, 420 contra 490)
queda un poco más cerca del modelo sin balancear, y la genérica (490 contra 490, con reposición) cae junto a
class weights y SMOTE, que es lo esperable porque las tres igualan exactamente el peso de las dos clases. El AUC casi no cambia, porque el AUC no depende del umbral ni del balanceo.

Y la fila que vale la conversación: el modelo sin balancear con umbral 0,30 tiene recall 0,856 y precision
0,513, casi lo mismo que class weights (0,856 y 0,527). Para una logística, **balancear y mover el umbral
son dos caminos al mismo lugar**. La diferencia es que el balanceo desplaza las probabilidades (la $p$
promedio en testing sube de 0,33 a 0,48 cuando la tasa real es 0,30: el modelo deja de estar calibrado) y el
umbral no. Cuándo sí balancear: cuando la minoritaria es tan escasa que el modelo ni siquiera aprende a
distinguirla y el AUC mismo mejora al ponderar, o cuando el algoritmo no entrega una probabilidad que se
pueda umbralizar.



## 10. Cuánto confiar en una métrica: variación muestral, validación cruzada e intervalos

Todo lo anterior se midió en **una** muestra de testing de 300 clientes. Si el banco hubiera sacado otros
300, ¿el AUC seguiría siendo 0,82 y el recall 0,57? La forma más directa de responder es hacerlo: repetir la
partición training/testing muchas veces con distintas semillas, reentrenar y medir cada vez. Eso se llama
**variación muestral**, y es la razón de existir de todo lo que sigue.


In [ ]:

X_all, y_all = S[XCOLS].values, S["y"].values
def entrenar_medir(semilla):
    Xa, Xb, ya, yb = train_test_split(X_all, y_all, test_size=0.3, stratify=y_all, random_state=semilla)
    m = LogisticRegression(penalty=None, max_iter=5000).fit(Xa, ya)
    p = m.predict_proba(Xb)[:, 1]
    return dict(semilla=semilla, AUC=roc_auc_score(yb, p), recall=recall_score(yb, (p > 0.5).astype(int)), precision=precision_score(yb, (p > 0.5).astype(int)))

reps = pd.DataFrame([entrenar_medir(s) for s in range(30)])
print(reps.describe().loc[["mean", "std", "min", "max"]].round(3))
fig = px.histogram(reps, x="AUC", nbins=15, title="30 particiones distintas, 30 AUC distintos (el nuestro fue 0,822)")
fig.add_vline(x=auc_sklearn, line_dash="dash"); fig.update_layout(height=340); fig.show()



El AUC se mueve entre 0,72 y 0,81 según qué clientes cayeron en testing, con promedio 0,77, y el recall entre
0,36 y 0,56. Nuestro 0,82 y nuestro recall de 0,57 quedaron por encima de las 30 particiones: la muestra de
testing que nos tocó fue amable, y ninguna de las conclusiones anteriores debería apoyarse en que el AUC "es"
0,82. Dos conclusiones. Una métrica sobre una sola partición
es una **estimación con ruido**, y hay que reportarla con su incertidumbre. Y comparar dos modelos por una
diferencia de un punto de AUC en una sola partición no dice nada: la diferencia es más chica que el ruido.

### Validación cruzada

Repetir particiones al azar desperdicia datos (cada cliente sale muchas veces en testing y otras ninguna).
La **validación cruzada** de $k$ pliegues ordena la idea: divide training en $k$ partes iguales, entrena con
$k - 1$ y mide en la que quedó fuera, rota $k$ veces, y promedia. Cada cliente se usa para medir exactamente
una vez. Con `RepeatedStratifiedKFold` se repite el proceso con varios barajados, y la desviación estándar
entre pliegues es la medida de incertidumbre.

Además de estimar el desempeño, la validación cruzada es la forma honesta de **elegir** cosas: el umbral, la
$\lambda$ de la regularización, el número de variables. Se eligen mirando solo training (por dentro, con los
pliegues), y testing queda intacto para el veredicto final. Elegir el umbral mirando testing, como hicimos en
la sección 7 para explicar la idea, es una forma suave de sobreajuste.


In [ ]:

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=0)
aucs, costos_por_umbral = [], []
for i_tr, i_va in cv.split(X_tr, y_tr):
    m = LogisticRegression(penalty=None, max_iter=5000).fit(X_tr[i_tr], y_tr[i_tr])
    p_va = m.predict_proba(X_tr[i_va])[:, 1]
    aucs.append(roc_auc_score(y_tr[i_va], p_va))
    t = metricas_por_umbral(y_tr[i_va], p_va, umbrales)
    costos_por_umbral.append((t["FP"] * c_FP + t["FN"] * c_FN).values)
aucs = np.array(aucs); costos_por_umbral = np.array(costos_por_umbral)
print(f"AUC por validación cruzada (5 pliegues x 10 repeticiones, solo training): {aucs.mean():.3f} +/- {aucs.std():.3f}   (en nuestro testing fue {auc_sklearn:.3f})")
costo_medio = pd.Series(costos_por_umbral.mean(axis=0), index=umbrales)
print("Umbral con menor costo medio en validación cruzada:", costo_medio.idxmin(), " (fórmula: 0,167; tabla sobre testing: 0,25)")
print(costo_medio.round(1).to_string())



### Intervalos de confianza por bootstrap

Para poner una barra de error a la métrica de testing sin salir de testing, se usa el **bootstrap**:
remuestrear los 300 clientes de testing con reposición (algunos salen dos veces, otros ninguna), recalcular la
métrica, y repetir mil veces. Los percentiles 2,5 y 97,5 de esas mil métricas son un intervalo de confianza
del 95%. Es la herramienta más simple y más general que existe para esto: funciona para el AUC, el recall, el
costo o el umbral óptimo, sin fórmulas.


In [ ]:

rng_bs = np.random.default_rng(0)
B = 1000
bs = {"AUC": [], "recall (umbral 0,5)": [], "precision (umbral 0,5)": [], "umbral óptimo (tabla)": []}
for _ in range(B):
    idx = rng_bs.integers(0, len(y_te), len(y_te))
    yb, pb = y_te[idx], p_te[idx]
    if yb.min() == yb.max():
        continue
    bs["AUC"].append(roc_auc_score(yb, pb))
    pred = (pb > 0.5).astype(int)
    bs["recall (umbral 0,5)"].append(recall_score(yb, pred)); bs["precision (umbral 0,5)"].append(precision_score(yb, pred))
    t = metricas_por_umbral(yb, pb, umbrales)
    bs["umbral óptimo (tabla)"].append(float(t.loc[(t["FP"] * c_FP + t["FN"] * c_FN).idxmin(), "umbral"]))
resumen = pd.DataFrame({k: dict(estimacion_original=v0, ic_2_5=np.percentile(v, 2.5), ic_97_5=np.percentile(v, 97.5))
                        for (k, v), v0 in zip(bs.items(), [auc_sklearn, recall_score(y_te, (p_te > .5).astype(int)), precision_score(y_te, (p_te > .5).astype(int)), 0.25])}).T
print(resumen.round(3))
fig = px.histogram(x=bs["umbral óptimo (tabla)"], nbins=20, title="Bootstrap: dónde cae el umbral óptimo de la tabla si cambio la muestra de testing")
fig.update_layout(xaxis_title="umbral óptimo", height=340); fig.show()



Antes, una lectura de la validación cruzada: estimó un AUC de 0,74, bien por debajo del 0,82 de testing. Ese
0,74 es la expectativa honesta para un cliente nuevo; el 0,82 fue suerte de muestra, como ya vimos. Y el
umbral de menor costo medio por validación cruzada cayó en 0,15, cerca de la fórmula (0,167) y lejos del 0,25
de la tabla sobre testing.

El AUC de testing no es 0,82: es "entre 0,76 y 0,87 con 95% de confianza". El recall no es 0,57: es "entre
0,47 y 0,67". Y el umbral óptimo de la tabla no es 0,25: se reparte entre 0,15 y 0,35 según qué clientes
caen en la muestra, que es exactamente lo que la sección 7 anticipaba. Con eso se cierra el círculo: la
fórmula (0,167) y el umbral de validación cruzada (0,15) quedan dentro del intervalo, y la diferencia con 0,25
no es evidencia de nada.



## 11. Lo que hay que llevarse

1. Una recta no sirve para un sí o no porque no tiene tope; la **función logística** la lleva a $(0, 1)$ y
   convierte el puntaje en una probabilidad. El umbral que colapsa esa probabilidad en una decisión es una
   elección aparte.
2. Un modelo de probabilidades se entrena minimizando la **entropía cruzada**, que castiga estar seguro y
   equivocarse. Tiene un solo mínimo: Solver y scikit-learn llegan al mismo lugar.
3. El acierto esconde lo que importa. La **matriz de confusión** y, sobre todo, **recall** (eficacia) y
   **precision** (eficiencia) de la clase que interesa son lo que hay que mirar, y hay un trade-off entre
   ellas que depende del negocio.
4. La palanca del trade-off es el **umbral**. La curva ROC es la tabla de umbrales dibujada; el AUC la resume
   sin fijar umbral. Con costos, el umbral óptimo se calcula: por fórmula si el modelo está calibrado, por
   tabla siempre, y las dos respuestas difieren por calibración o por ruido muestral.
5. La **regularización** se agrega a la logística con un término más en la función de costo. En esta base no
   mejora la predicción, y decirlo es parte del método.
6. El **desbalanceo** se corrige ponderando o remuestreando; para una logística, casi todo eso equivale a
   mover el umbral, y el balanceo descalibra las probabilidades.
7. Una métrica en una partición es una estimación con ruido. La **validación cruzada** la estima mejor y es
   la forma honesta de elegir umbral o $\lambda$; el **bootstrap** le pone un intervalo. Sin eso, una
   diferencia de un punto entre dos modelos no significa nada.



## Ejercicios

**Ejercicio 1.** Repite la preparación con las 20 columnas originales (todas las categóricas a dummies con
`pd.get_dummies(drop_first=True)`) y entrena la misma logística sin regularizar. ¿Mejora el AUC de testing?
Antes de mirar el resultado, escribe cuánto esperas que mejore y por qué; después compara con el intervalo de
bootstrap de la sección 10.

**Ejercicio 2.** Cambia los costos a $c_{FN} = 10$ y $c_{FP} = 1$ (un banco muy conservador). Calcula el
umbral por fórmula, por tabla sobre testing y por validación cruzada. ¿Cuántos buenos pagadores rechaza el
banco con ese umbral, y cuántos malos deja pasar?

**Ejercicio 3.** Con lasso, recorre $\lambda$ en `[1, 3, 10, 30]` usando validación cruzada de 5 pliegues
sobre training (no testing) para elegir la mejor. Reporta cuántas variables quedan y el AUC de testing del
modelo elegido, con su intervalo de bootstrap. ¿Es distinguible del modelo con las 18 variables?

**Ejercicio 4.** Simula un desbalanceo severo: quédate en training con solo 40 de los 210 malos pagadores
(al azar) y los 490 buenos. Entrena sin balancear y con `class_weight="balanced"`. Compara AUC y recall de la
clase 1 en el testing original. ¿Con 8% de minoritaria, el balanceo sigue siendo equivalente a mover el
umbral, o ahora sí mejora el AUC?



## Soluciones


In [ ]:

# SOLUCIÓN 1: las 20 columnas
G = german.drop(columns=["id", "class"]).copy()
num20 = G.select_dtypes(exclude="object").columns.tolist()
X20 = pd.get_dummies(G, drop_first=True).astype(float)
X20[num20] = (X20[num20] - X20.loc[tr, num20].mean()) / X20.loc[tr, num20].std(ddof=0)
m20 = LogisticRegression(penalty=None, max_iter=10000).fit(X20[tr].values, y_tr)
p20 = m20.predict_proba(X20[te].values)[:, 1]
print("Columnas:", X20.shape[1], "  AUC testing con 20 variables:", round(roc_auc_score(y_te, p20), 3), " (con 9:", round(auc_sklearn, 3), f"; intervalo bootstrap de las 9: {np.percentile(bs['AUC'], 2.5):.3f} a {np.percentile(bs['AUC'], 97.5):.3f})")
print("Lectura: la mejora, si la hay, está dentro del ruido muestral. Más columnas no es más información útil por sí mismo.")


In [ ]:

# SOLUCIÓN 2: costos 10 y 1
cFN2, cFP2 = 10, 1
u_formula = cFP2 / (cFP2 + cFN2)
t2 = metricas_por_umbral(y_te, p_te, umbrales); t2["costo"] = t2["FP"] * cFP2 + t2["FN"] * cFN2
u_tabla = t2.loc[t2["costo"].idxmin(), "umbral"]
costos2 = []
for i_tr, i_va in cv.split(X_tr, y_tr):
    m = LogisticRegression(penalty=None, max_iter=5000).fit(X_tr[i_tr], y_tr[i_tr])
    t = metricas_por_umbral(y_tr[i_va], m.predict_proba(X_tr[i_va])[:, 1], umbrales)
    costos2.append((t["FP"] * cFP2 + t["FN"] * cFN2).values)
u_cv = pd.Series(np.mean(costos2, axis=0), index=umbrales).idxmin()
print("Umbral por fórmula:", round(u_formula, 3), " por tabla en testing:", u_tabla, " por validación cruzada:", u_cv)
fila = t2.loc[t2.umbral == u_cv].iloc[0]
print(f"Con el umbral de validación cruzada ({u_cv}) en testing: rechaza {int(fila.FP)} buenos pagadores de 210 y deja pasar {int(fila.FN)} malos de 90.")


In [ ]:

# SOLUCIÓN 3: lasso con lambda elegida por validación cruzada
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
res3 = {}
for lam in [1, 3, 10, 30]:
    a = []
    for i_tr, i_va in cv5.split(X_tr, y_tr):
        m = LogisticRegression(penalty="l1", C=1 / lam, solver="saga", max_iter=50000, tol=1e-8).fit(X_tr[i_tr], y_tr[i_tr])
        a.append(roc_auc_score(y_tr[i_va], m.predict_proba(X_tr[i_va])[:, 1]))
    res3[lam] = np.mean(a)
mejor = max(res3, key=res3.get)
m3 = LogisticRegression(penalty="l1", C=1 / mejor, solver="saga", max_iter=50000, tol=1e-8).fit(X_tr, y_tr)
p3 = m3.predict_proba(X_te)[:, 1]
bs3 = [roc_auc_score(y_te[i], p3[i]) for i in (rng_bs.integers(0, len(y_te), len(y_te)) for _ in range(1000)) if y_te[i].min() != y_te[i].max()]
print("AUC por validación cruzada según lambda:", {k: round(v, 3) for k, v in res3.items()}, " -> mejor lambda:", mejor)
print("Variables que quedan:", int((np.abs(m3.coef_[0]) > 1e-6).sum()), "de 18.  AUC testing:", round(roc_auc_score(y_te, p3), 3),
      f" IC 95% bootstrap: {np.percentile(bs3, 2.5):.3f} a {np.percentile(bs3, 97.5):.3f}")
print("Lectura: el modelo con menos variables no se distingue del completo; los intervalos se superponen casi por completo.")


In [ ]:

# SOLUCIÓN 4: desbalanceo severo (8% de malos en training)
rng4 = np.random.default_rng(4)
malos = np.where(y_tr == 1)[0]; buenos = np.where(y_tr == 0)[0]
idx4 = np.r_[rng4.choice(malos, 40, replace=False), buenos]
X4, y4 = X_tr[idx4], y_tr[idx4]
print("Training severo:", len(y4), "clientes,", int(y4.sum()), "malos (", round(y4.mean() * 100, 1), "%)")
filas4 = []
for nombre, m in [("sin balancear", LogisticRegression(penalty=None, max_iter=5000)), ("class weights balanced", LogisticRegression(penalty=None, max_iter=5000, class_weight="balanced"))]:
    m.fit(X4, y4); p4 = m.predict_proba(X_te)[:, 1]
    for u in (0.5, 0.3, 0.1):
        pred = (p4 > u).astype(int)
        filas4.append(dict(modelo=nombre, umbral=u, AUC=roc_auc_score(y_te, p4), recall_1=recall_score(y_te, pred), precision_1=precision_score(y_te, pred, zero_division=0), p_promedio=p4.mean()))
print(pd.DataFrame(filas4).round(3).to_string(index=False))
print("Lectura: el AUC es el mismo con y sin balanceo (la recta ordena igual); lo que cambia es dónde queda el umbral 0,5. Con 8% de minoritaria, sin balancear hay que bajar el umbral a 0,1 para atrapar a alguien; balancear lo hace por ti. Sigue siendo equivalente a mover el umbral, pero el modelo balanceado es más cómodo de usar.")
